# CLIPS Expert System

A refresher on **CLIPS** (C Language Integrated Production System) — NASA's classic
forward-chaining rule engine — driven from Python via **clipspy**.

**Domain:** Symbolic AI & Logic · **runnable:** yes · _needs `clipspy`_

## 1. What & Why

**CLIPS** is a production-rule (forward-chaining) expert-system shell, written in C at
NASA's Johnson Space Center in 1985 and still maintained. You encode knowledge as
**facts** (data) and **rules** (`IF condition THEN action`); the engine repeatedly
matches rules against the fact base and fires them until nothing more applies. It is the
canonical, fast, embeddable implementation of the OPS5-style production system.

**The problem it solves.** When domain logic is a large, shifting pile of *independent*
`if/then` policies — diagnostics, configuration, eligibility, compliance, monitoring —
hand-written nested `if` statements rot fast: order matters, adding a case means surgery,
and the "knowledge" is buried in control flow. A rule engine inverts this: you declare
each rule in isolation and the engine decides *what* fires and *when* via pattern
matching. Knowledge becomes additive — drop in a new rule, no rewiring.

**Reach for it when:**

- The logic is naturally "lots of small rules over evolving facts" and domain experts
  (not just programmers) need to read/extend it.
- You want **data-driven** control: the next action depends on the current fact base,
  not a fixed call sequence.
- You need a small, fast, dependency-free engine you can embed (CLIPS is C; clipspy wraps it).

**Don't reach for it when** the problem is really search/optimization (use a SAT/SMT/CP
solver — see `z3-smt`, `minizinc`), probabilistic (`problog`), or just a handful of
branches a plain function expresses more clearly. Forward chaining also explores *all*
derivable consequences; if you only want to answer one query, backward chaining
(Prolog — see `swi-prolog`) is often a better fit.

## 2. Mental Model

Think of a **bulletin board (working memory) watched by a room of specialists (rules)**:

```
            ┌──────────────── Working Memory (facts) ────────────────┐
            │  (symptom (name fever))   (symptom (name aches)) ...    │
            └────────────────────────────┬───────────────────────────┘
                                          │ pattern match (Rete network)
                                          ▼
                          ┌───────────────────────────────┐
   each rule = a          │  AGENDA  (matched rules ready  │
   specialist watching ──►│  to fire, ordered by salience) │
   for its patterns       └───────────────┬───────────────┘
                                          │ pick top, FIRE its RHS
                                          ▼
                         action: assert / retract / modify facts,
                         print, call out  →  changes WM  →  re-match
```

The engine runs a **recognize–act cycle**: (1) match every rule's left-hand side (LHS)
against working memory, (2) put each fully-matched rule + its bindings on the **agenda**,
(3) fire the highest-priority one, executing its right-hand side (RHS), which usually
changes working memory — then loop. It halts when the agenda is empty.

Two things to internalize: rules are **not** called, they are *triggered by data*; and
matching is incremental — CLIPS uses the **Rete algorithm** (see `rete-algorithm`) to
remember partial matches so each small fact change is cheap, not a full re-scan.

## 3. Key Concepts

| Term | What it is |
|------|-----------|
| **Fact** | A unit of data in working memory. Either an *ordered* fact `(temperature high)` or a *templated* fact `(animal (name dog) (trait has-hair))`. |
| **`deftemplate`** | A schema (named slots) for structured facts — the rule-engine analogue of a struct/record. |
| **Rule (`defrule`)** | `(defrule name LHS => RHS)`. The **LHS** is a set of patterns/conditions; the **RHS** is actions run when the LHS matches. |
| **Pattern matching & binding** | LHS patterns match facts; `?x` variables capture slot values and must stay consistent across patterns (a join). |
| **Working memory** | The current set of all asserted facts. Rules read it; RHS actions (`assert`/`retract`/`modify`) change it. |
| **Agenda** | The ordered list of rule *activations* (matched rule + bindings) waiting to fire. |
| **Salience** | An integer priority (default 0) overriding agenda order; higher fires first. Use sparingly. |
| **Conflict resolution** | How ties on the agenda break (salience, then recency/specificity). |
| **Recognize–act cycle** | match → agenda → fire one → repeat until agenda empty (`(run)`). |
| **Rete network** | The compiled match graph that makes incremental matching fast. |
| **Truth maintenance** | Facts asserted *as a logical consequence* (via `logical` CEs) are auto-retracted when their support disappears. |

## 4. Setup

CLIPS itself is C, but **clipspy** ships a self-contained wheel (the CLIPS C library is
bundled) — no separate native install, no compiler needed on common platforms:

```bash
pip install clipspy
```

The cell below installs it only if it's missing, so the notebook runs in a fresh kernel.

In [1]:
# Idempotent install: only pip-install clipspy if it isn't already importable.
import importlib.util, sys, subprocess

if importlib.util.find_spec("clips") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "clipspy"], check=True)

import clips
print("clipspy", clips.__version__)

clipspy 1.0.6


## 5. Worked Examples

### Example 1 — facts, a template, and a rule that fires

The "hello world" of forward chaining: assert a structured fact, and watch a rule derive
a new one. We declare a `deftemplate` (schema), one rule that says *anything with hair is
warm-blooded*, assert a `dog`, and `run()` the engine. Note we never *call* the rule —
asserting the fact is what triggers it.

In [2]:
import clips

env = clips.Environment()

# A schema for structured facts, and a rule over it.
env.build("(deftemplate animal (slot name) (slot trait))")
env.build("""
(defrule mammal-is-warm-blooded
  (animal (name ?n) (trait has-hair))      ; LHS: match any haired animal, bind its name
  =>
  (assert (animal (name ?n) (trait warm-blooded)))   ; RHS: derive a new fact
  (printout t ?n " is warm-blooded" crlf))
""")

# Drop a fact into working memory and let the engine run the recognize-act cycle.
env.assert_string("(animal (name dog) (trait has-hair))")
fired = env.run()                 # returns the number of rules fired
print("rules fired:", fired)

print("\nworking memory:")
for fact in env.facts():
    print(" ", fact)

rules fired: 1

working memory:
  (animal (name dog) (trait has-hair))
  (animal (name dog) (trait warm-blooded))


The rule fired once: CLIPS matched the haired `dog`, bound `?n = dog`, and asserted the
derived `warm-blooded` fact (now visible in working memory). That's the whole loop —
**data in, consequences out**.

### Example 2 — a tiny diagnostic system: salience, negation, and reading results back

A more expert-system-shaped example. Symptoms are facts; rules infer a diagnosis. We use:

- **salience** to make the high-confidence `flu` rule outrank `cold`,
- a **negated condition** `(not (diagnosis (disease flu)))` so we only diagnose a cold
  when flu hasn't already been concluded,
- and we **read the structured result facts back into Python** for downstream use.

In [3]:
env = clips.Environment()

env.build("(deftemplate symptom   (slot name))")
env.build("(deftemplate diagnosis (slot disease) (slot confidence))")

# Higher salience => this rule's activation sits at the top of the agenda.
env.build("""
(defrule flu (declare (salience 10))
  (symptom (name fever))
  (symptom (name aches))
  =>
  (assert (diagnosis (disease flu) (confidence high))))
""")

# Only conclude "cold" if flu was NOT already diagnosed (negated conditional element).
env.build("""
(defrule cold
  (symptom (name runny-nose))
  (not (diagnosis (disease flu)))
  =>
  (assert (diagnosis (disease cold) (confidence medium))))
""")

for s in ("fever", "aches", "runny-nose"):
    env.assert_string(f"(symptom (name {s}))")

env.run()

# Pull structured facts back out as ordinary Python dicts.
results = [
    {"disease": f["disease"], "confidence": f["confidence"]}
    for f in env.facts()
    if f.template.name == "diagnosis"
]
print("diagnoses:", results)

diagnoses: [{'disease': 'flu', 'confidence': 'high'}]


Even though the patient has a runny nose, **no cold is diagnosed**: `flu` has higher
salience, fires first, and its derived `(diagnosis (disease flu) ...)` fact makes the
`cold` rule's `(not ...)` condition false. This is the engine doing conflict resolution
and non-monotonic reasoning for you — and clipspy hands the result facts straight back as
Python data you can serialize, store, or feed onward.

### Example 3 (optional) — load a knowledge base from a `.clp` file

Real systems keep rules in `.clp` files and `(load ...)` them. This cell is gated behind
an env-var so it never fails in a fresh kernel; it shows the call shape.

In [4]:
import os, textwrap, tempfile

if os.getenv("CLIPS_RUN_FILE_EXAMPLE"):
    kb = textwrap.dedent("""
        (defrule greet => (printout t "knowledge base loaded" crlf))
    """)
    with tempfile.NamedTemporaryFile("w", suffix=".clp", delete=False) as fh:
        fh.write(kb)
        path = fh.name
    env = clips.Environment()
    env.load(path)        # parse + add all constructs from the file
    env.reset()           # assert any (deffacts) and initial facts
    env.run()
else:
    print("Set CLIPS_RUN_FILE_EXAMPLE=1 to run the .clp file-loading example.")
    print("Shape: env.load('rules.clp'); env.reset(); env.run()")

Set CLIPS_RUN_FILE_EXAMPLE=1 to run the .clp file-loading example.
Shape: env.load('rules.clp'); env.reset(); env.run()


## 6. Gotchas & Pitfalls

- **Forgetting `(run)`** (`env.run()`). Asserting facts only *schedules* activations on
  the agenda; nothing fires until you run the engine. A silent no-op usually means you
  built rules and facts but never ran.
- **`reset()` wipes working memory** and re-asserts only `(deffacts)` and the initial
  fact. If you `assert` facts from Python and then call `reset()`, those facts are gone.
- **Refraction & infinite loops.** A rule won't re-fire on the *same* matched facts, but a
  rule whose RHS asserts a fact its own LHS matches (with fresh facts each time) can loop
  forever. Watch self-triggering rules; use `modify`/`retract` to consume what you assert.
- **Ordered vs templated facts.** `(point 3 4)` (ordered) and
  `(point (x 3) (y 4))` (templated) are different worlds. You can't pattern-match a
  templated fact with ordered syntax; pick one per fact type and declare a `deftemplate`.
- **Salience is a smell in bulk.** A few priorities are fine; encoding your whole control
  flow in salience numbers recreates the spaghetti you adopted a rule engine to escape.
  Prefer expressing ordering through facts/conditions (control facts, phases).
- **`build()` parse errors are terse.** clipspy raises on malformed constructs; mismatched
  parentheses or an unknown slot give a generic error. Build one construct at a time while
  developing so you know which string failed.
- **One `Environment` is one isolated world.** Templates, rules, and facts live inside an
  `Environment`. Spinning up a second `Environment` shares nothing — handy for isolation,
  surprising if you expected globals.
- **Strings vs symbols.** `fever` is a *symbol*; `"fever"` is a *string* — they don't
  match each other. Be consistent, especially when interpolating Python values.

## 7. When to Use vs Alternatives

| Option | Paradigm | Use it when… | vs CLIPS |
|--------|----------|--------------|----------|
| **CLIPS / clipspy** | Forward chaining (production rules) | Many independent `if/then` policies over evolving facts; embeddable, fast, mature | The baseline here |
| **Prolog** (`swi-prolog`) | Backward chaining (goal-directed) | You ask *queries* and want answers/proofs, recursion, unification | Prolog is goal-driven & answers one question; CLIPS derives *all* consequences as data changes |
| **Datalog** (`datalog`) | Bottom-up deduction, no side effects | Pure declarative recursive queries over relations; guaranteed termination | Datalog is a restricted, decidable subset; CLIPS adds side-effecting RHS actions & no termination guarantee |
| **Drools / business-rule engines** | Forward chaining on the JVM | Enterprise Java stack, BPM integration, web rule authoring | Same Rete idea, heavier; CLIPS is tiny, C-native, no JVM |
| **`experta` / `pyknow`** | Pure-Python CLIPS-like | You want native-Python rules, no C dependency | clipspy wraps the real (faster, battle-tested) CLIPS engine; experta is pure Python, less maintained |
| **SMT / CP solvers** (`z3-smt`, `minizinc`) | Constraint solving / search | Optimization, scheduling, "find values satisfying constraints" | Solvers *search* a space; CLIPS *fires* rules — wrong tool if you're optimizing |
| **Plain `if/elif`** | Imperative | A handful of fixed, ordered branches | No engine to learn; but doesn't scale as rule count and interactions grow |

**Rule of thumb:** if you can name dozens of independent "when *this* situation, do *that*"
policies that change over time and interact through shared data, a forward-chaining engine
like CLIPS earns its keep. If you're answering a single query, optimizing, or you have five
fixed branches, reach for Prolog, a solver, or just a function.

## 8. Resources

- **CLIPS official site & downloads** — http://www.clipsrules.net/
- **CLIPS Basic Programming Guide (the canonical reference manual, PDF)** —
  https://clipsrules.sourceforge.io/documentation/v640/bpg.pdf
- **clipspy documentation (the Python binding used here)** —
  https://clipspy.readthedocs.io/
- **clipspy source (GitHub)** — https://github.com/noxdafox/clipspy
- **Giarratano & Riley, *Expert Systems: Principles and Programming*** — the standard
  textbook behind CLIPS' design; good for the theory of production systems and Rete.
- Cross-links in this library: `rete-algorithm` (how matching is made fast),
  `swi-prolog` (backward chaining), `datalog` (pure deduction), `pyke` (another Python
  rule engine).